In [ ]:
import os

# TileLang 在导入时读取 tracing 配置；命令行环境变量可以覆盖默认值。
# os.environ.setdefault("TL_LOWER_TRACE", "html")
os.environ.setdefault("TILELANG_DISABLE_CACHE", "1")

In [ ]:
import torch
import tilelang
import tilelang.ascend.language as T

N = 2

@T.prim_func
def my_kernel(
    A: T.Tensor((N,), T.float32), # type: ignore
    B: T.Tensor((N,), T.float32), # type: ignore
):
    blocks = N
    with T.Kernel(blocks) as bx:
        B[bx] = A[bx] + 1
        T.print(msg="test")


jit_kernel = tilelang.compile(my_kernel, out_idx=[-1])
# print(jit_kernel.get_kernel_source())

a = torch.ones(N, dtype=torch.float32).npu()
b = jit_kernel(a)
torch.npu.synchronize()  # 等待计算完成，确保结果正确
print("Input A:", a)
print("Result:", b)